# Packages

In [1]:
# pip install pdfplumber
# !pip uninstall -y transformers tokenizers huggingface_hub
# !pip install "transformers==4.52.2" sentencepiece accelerate

In [1]:
from pathlib import Path
import re
import nltk
import pdfplumber
from transformers import pipeline


for resource in ["punkt", "punkt_tab"]:
    try:
        nltk.data.find(f"tokenizers/{resource}")
    except LookupError:
        nltk.download(resource)

from nltk.tokenize import sent_tokenize

I0000 00:00:1777993011.570019  141761 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777993011.650732  141761 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777993013.996060  141761 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


# COnfiguration 

In [2]:
PDF_PATH = Path("google_terms_of_service_en_in.pdf")
OUTPUT_TEXT_FILE = Path("extracted_text.txt")

SUMMARIZATION_MODEL = "t5-small"
QUESTION_GENERATION_MODEL = "valhalla/t5-base-qg-hl"
QA_MODEL = "deepset/roberta-base-squad2"

MAX_PASSAGE_WORDS = 200
MIN_QUESTIONS_PER_PASSAGE = 3

# Utilities functions

In [3]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    """
    Extract clean text from a PDF file.
    """
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF file not found: {pdf_path}")

    pages_text = []

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                pages_text.append(text)

    full_text = "\n".join(pages_text)

    # Clean extra spaces
    full_text = re.sub(r"\s+", " ", full_text).strip()

    return full_text

def save_text(text: str, output_path: Path) -> None:
    """
    Save extracted text to a file.
    """
    output_path.write_text(text, encoding="utf-8")

def load_text(path: Path) -> str:
    """
    Load text from a file.
    """
    if not path.exists():
        raise FileNotFoundError(f"Text file not found: {path}")

    return path.read_text(encoding="utf-8")


def split_into_passages(text: str, max_words: int = 200) -> list[str]:
    """
    Split a long document into passages of approximately max_words words.
    """
    sentences = sent_tokenize(text)

    passages = []
    current_passage = []
    current_word_count = 0

    for sentence in sentences:
        sentence_word_count = len(sentence.split())

        if current_word_count + sentence_word_count <= max_words:
            current_passage.append(sentence)
            current_word_count += sentence_word_count
        else:
            passages.append(" ".join(current_passage))
            current_passage = [sentence]
            current_word_count = sentence_word_count

    if current_passage:
        passages.append(" ".join(current_passage))

    return passages

# Load Models

In [4]:
summarizer = pipeline(
    "summarization",
    model=SUMMARIZATION_MODEL,
    tokenizer=SUMMARIZATION_MODEL,
    framework="pt"
)

qg_pipeline = pipeline(
    "text2text-generation",
    model=QUESTION_GENERATION_MODEL,
    tokenizer=QUESTION_GENERATION_MODEL,
    framework="pt"
)

qa_pipeline = pipeline(
    "question-answering",
    model=QA_MODEL,
    tokenizer=QA_MODEL,
    framework="pt"
)

Device set to use cpu
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Device set to use cpu
Device set to use cpu


# Extract and preview document



In [5]:
document_text = extract_text_from_pdf(PDF_PATH)
save_text(document_text, OUTPUT_TEXT_FILE)

print("Number of characters:", len(document_text))
print("Preview:\n")
print(document_text[:500])

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Number of characters: 26779
Preview:

GOOGLE TERMS OF SERVICE Effective May 22, 2024 | Archived versions What’s covered in these terms We know it’s tempting to skip these Terms of Service, but it’s important to establish what you can expect from us as you use Google services, and what we expect from you. These Terms of Service re ect the way Google’s business works, the laws that apply to our company, and certain things we’ve always believed to be true. As a result, these Terms of Service help de ne Google’s relationship with you as


# Summarize document

In [7]:
def summarize_text(text: str, max_input_chars: int = 3000) -> str:
    """
    Summarize the beginning of a document.
    T5-small cannot process very long documents, so we truncate safely.
    """
    text = text[:max_input_chars]

    result = summarizer(
        text,
        max_length=150,
        min_length=30,
        do_sample=False
    )

    return result[0]["summary_text"]

summary = summarize_text(document_text)

print("Summary:\n")
print(summary)

Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summary:

these Terms of Service help dene the relationship between you and Google . these terms reect the way Google’s business works, the laws that apply to our company, and certain things we’ve always believed to be true . if you’re under the age required to manage your own Google Account, you must have your parent or legal guardian read these terms with you .


# Generate questions

In [8]:
def generate_questions(passage: str, min_questions: int = 3) -> list[str]:
    """
    Generate questions from one passage.
    """
    input_text = f"generate questions: {passage}"

    result = qg_pipeline(
        input_text,
        max_length=128,
        do_sample=False
    )

    generated_text = result[0]["generated_text"]

    questions = [
        q.strip()
        for q in generated_text.split("<sep>")
        if q.strip()
    ]

    # Fallback if too few questions are generated
    if len(questions) < min_questions:
        passage_sentences = sent_tokenize(passage)

        for i in range(0, len(passage_sentences), 2):
            if len(questions) >= min_questions:
                break

            short_context = " ".join(passage_sentences[i:i+2])

            if not short_context.strip():
                continue

            result = qg_pipeline(
                f"generate questions: {short_context}",
                max_length=128,
                do_sample=False
            )

            extra_questions = [
                q.strip()
                for q in result[0]["generated_text"].split("<sep>")
                if q.strip()
            ]

            questions.extend(extra_questions)

    # Remove duplicates while preserving order
    unique_questions = list(dict.fromkeys(questions))

    return unique_questions[:min_questions]

# Answer generated questions

In [9]:
def answer_questions_for_passage(passage: str, questions: list[str]) -> list[dict]:
    """
    Answer a list of questions using the given passage as context.
    """
    qa_results = []

    for question in questions:
        answer = qa_pipeline(
            question=question,
            context=passage
        )

        qa_results.append({
            "question": question,
            "answer": answer["answer"],
            "score": round(answer["score"], 4)
        })

    return qa_results

# full pipeline

In [12]:
passages = split_into_passages(document_text, max_words=MAX_PASSAGE_WORDS)

print(f"Number of passages: {len(passages)}")

Number of passages: 26


In [13]:
all_results = []

for idx, passage in enumerate(passages, start=1):
    questions = generate_questions(
        passage,
        min_questions=MIN_QUESTIONS_PER_PASSAGE
    )

    qa_results = answer_questions_for_passage(passage, questions)

    all_results.append({
        "passage_id": idx,
        "passage": passage,
        "qa": qa_results
    })

    print(f"\nPassage {idx}")
    print("-" * 60)
    print(passage[:500], "...")

    print("\nGenerated Q&A:")
    for item in qa_results:
        print(f"Q: {item['question']}")
        print(f"A: {item['answer']}  | score={item['score']}")
        print()

Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Passage 1
------------------------------------------------------------
GOOGLE TERMS OF SERVICE Effective May 22, 2024 | Archived versions What’s covered in these terms We know it’s tempting to skip these Terms of Service, but it’s important to establish what you can expect from us as you use Google services, and what we expect from you. These Terms of Service re ect the way Google’s business works, the laws that apply to our company, and certain things we’ve always believed to be true. As a result, these Terms of Service help de ne Google’s relationship with you as ...

Generated Q&A:
Q: What does the Google Terms of Service cover?
A: certain things we’ve always believed to be true  | score=0.0385

Q: What are the Google Terms of Service?
A: certain things we’ve always believed to be true  | score=0.0122

Q: What do these Terms of Service describe?
A: the way Google’s business works  | score=0.0357



Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Passage 2
------------------------------------------------------------
Besides these terms, we also publish a Privacy Policy. Although it’s not part of these terms, we encourage you to read it to better understand how you can update, manage, export, and delete your information. Terms Service provider Google services are provided by, and you’re contracting with: Google LLC organized under the laws of the State of Delaware, USA, and operating under the laws of the USA 1600 Amphitheatre Parkway Mountain View, California 94043 USA Age requirements If you’re under the a ...

Generated Q&A:
Q: What does Google LLC do with your information?
A: update, manage, export, and delete  | score=0.0004

Q: What is the Privacy Policy?
A: how you can update, manage, export, and delete your information  | score=0.0832

Q: What is the name of Google LLC?
A: Google LLC  | score=0.0141



Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


Passage 24
------------------------------------------------------------
If you don’t agree to the new terms, you should remove your content and stop using the services. You can also end your relationship with us at any time by closing your Google Account. DEFINITIONS a liate An entity that belongs to the Google group of companies, which means Google LLC and its subsidiaries, including the following companies that provide consumer services in the EU: Google Ireland Limited, Google Commerce Limited, and Google Dialer Inc. business user An individual or entity who is n ...

Generated Q&A:
Q: What is the definition of a company that belongs to the Google group of companies?
A: Google LLC  | score=0.3785

Q: What should you do if you don't agree to the new terms?
A: remove your content and stop using the services  | score=0.7549

Q: What does aliate mean?
A: An entity that belongs to the Google group of companies  | score=0.1143



Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Passage 25
------------------------------------------------------------
intellectual property rights (IP rights) Rights over the creations of a person’s mind, such as inventions (patent rights); literary and artistic works (copyright); designs (design rights); and symbols, names, and images used in commerce (trademarks). IP rights may belong to you, another individual, or an organization. liability Losses from any type of legal claim, whether the claim is based on a contract, tort (including negligence), or other reason, and whether or not those losses could have be ...

Generated Q&A:
Q: What is the definition of a trademark?
A: symbols, names, and images used in commerce  | score=0.011

Q: What are intellectual property rights?
A: Rights over the creations of a person’s mind  | score=0.5721

Q: What is the definition of a legal entity?
A: organization  | score=0.3266



Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Passage 26
------------------------------------------------------------
your content Things that you create, upload, submit, store, send, receive, or share using our services, such as: Docs, Sheets, and Slides you create blog posts you upload through Blogger reviews you submit through Maps videos you store in Drive emails you send and receive through Gmail pictures you share with friends through Photos travel itineraries that you share with Google ...

Generated Q&A:
Q: What are some of the things that you create, upload, submit, store or share using our services?
A: Docs, Sheets, and Slides  | score=0.9186

